# Product-Layer Valuation Engine Tests -- Fixed Accrued, the Three Atomic Index Cashflows, and Interest Rate Stream

Exercises, against a single shared three-component USD curve:

- `ValuationEngineProductFixedAccrued` -- pure fixed-coupon cashflow, no index.
- The three atomic index-linked cashflow engines built on the anchored-index analytics layer:
  `ValuationEngineProductOvernightIndexCompositeCashflow` (SOFR, daily-compounded),
  `ValuationEngineProductIBORIndexCashflow` (LIBOR-3M, single native-tenor period), and
  `ValuationEngineProductIBORCompoundingCashflow` (LIBOR-3M, 3 native-tenor periods
  geometrically compounded, both `SPREAD_EXCLUSIVE_COMPOUND` and `FLAT_COMPOUND`).
- `ValuationEngineProductInterestRateStream` -- a full leg (portfolio of the above), fixed and
  floating.

For every engine: PV at several `value_date`s (fully forward, on the payment date with
realized fixings, fully matured), `pv01()` against a closed form, and `get_risk()` against an
independent finite-difference (parallel-bump) estimate, checked block-by-block against the
curve's own component structure.

**Gotcha carried over from the other test notebooks in this directory:**
`model.discount_factor(index, date)` defaults to `calc_grad=False`, which flips
`requires_grad` to `False` *in place* on the curve component's shared state-data tensor -- a
plain (non-`calc_grad`) lookup run after an engine has already built its differentiable graph
on the same model would silently zero out that engine's later `.backward()`/`get_gradient()`.
Every closed-form comparison below passes `calc_grad=True` for that reason, even when the
result is only used for a print/assert.


In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import torch

from fixedincomelib import *
from fixedincomelib.yield_curve.valuation_engine import (
    ValuationEngineProductFixedAccrued,
    ValuationEngineProductOvernightIndexCompositeCashflow,
    ValuationEngineProductIBORIndexCashflow,
    ValuationEngineProductIBORCompoundingCashflow,
    ValuationEngineProductInterestRateStream,
    _to_float,
)
from fixedincomelib.product.linear_products import (
    ProductFixedAccrued,
    ProductOvernightIndexCompositeCashflow,
    ProductIBORIndexCashflow,
    ProductIBORCompoundingCashflow,
    ProductInterestRateStream,
)
from fixedincomelib.valuation.valuation_parameters import (
    ValuationParametersCollection,
    AnalyticValParam,
    FundingIndexParameter,
)

print("Fixed Income Library is loaded.")


Fixed Income Library is loaded.


## Build a shared USD curve

Three components: `SOFR-1B` (overnight projection, upward-sloping IFR), `USD-LIBOR-BBA-3M`
(IBOR projection, `REFERENCE`d off `SOFR-1B`), and `SOFR-1B-FLAT` (the discounting/funding
curve, also `REFERENCE`d off `SOFR-1B`, solved `BRENT` to a flat zero spread -- i.e. it tracks
`SOFR-1B` one-for-one, the way OIS discounting works in practice). Every engine below discounts
off `SOFR-1B-FLAT` and projects off either `SOFR-1B` or `USD-LIBOR-BBA-3M`.

`build_model(value_date, ...)` rebuilds the whole model at a given value date (and optionally
bumped IFR levels, for the finite-difference risk checks) -- there is no in-place "reprice as of
a different date" operation in this codebase, so every value-date comparison below constructs
its own model.


In [2]:
TENORS = ['3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']
FLAT_TENORS = ['1Y', '10Y', '30Y']

SOFR_IFR = [0.0430, 0.0432, 0.0435, 0.0440, 0.0445, 0.0448, 0.0450]
LIBOR_IFR = [0.0450, 0.0452, 0.0455, 0.0458, 0.0460, 0.0462, 0.0465]
FLAT_IFR = [0.0, 0.0, 0.0]

bm_list = [
    qfCreateBuildMethod('YC_OVERNIGHT_INDEX_ELEMENT', {
        'TARGET': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-IFR',
    }),
    qfCreateBuildMethod('YC_IBOR_ELEMENT', {
        'TARGET': 'USD-LIBOR-BBA-3M',
        'REFERENCE': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-LIBOR-BBA-3M-IFR',
    }),
    qfCreateBuildMethod('YC_FUNDING_ELEMENT', {
        'TARGET': 'SOFR-1B-FLAT',
        'REFERENCE': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-FLAT-IFR',
    }),
    qfCreateBuildMethod('YC_COMMON', {
        'TARGET': 'USD',
        'FUNDING PARAMETERS': 'SOFR-1B-FLAT',
        'SOLVER METHOD': 'BRENT',
    }),
]
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)


def _mkdf(index, values):
    df = pd.DataFrame(index=index)
    df['values'] = values
    return df


def build_data(sofr_ifr=None, libor_ifr=None, flat_ifr=None):
    return qfCreateDataCollection([
        qfCreateData1D('INSTANTANEOUS FORWARD RATE', 'USD-SOFR-OIS-1B-IFR',
                        _mkdf(TENORS, sofr_ifr if sofr_ifr is not None else SOFR_IFR)),
        qfCreateData1D('INSTANTANEOUS FORWARD RATE', 'USD-LIBOR-BBA-3M-IFR',
                        _mkdf(TENORS, libor_ifr if libor_ifr is not None else LIBOR_IFR)),
        qfCreateData1D('INSTANTANEOUS FORWARD RATE', 'USD-SOFR-OIS-1B-FLAT-IFR',
                        _mkdf(FLAT_TENORS, flat_ifr if flat_ifr is not None else FLAT_IFR)),
    ])


def build_model(value_date, sofr_bump=0.0, libor_bump=0.0, flat_bump=0.0):
    dc = build_data(
        sofr_ifr=[v + sofr_bump for v in SOFR_IFR],
        libor_ifr=[v + libor_bump for v in LIBOR_IFR],
        flat_ifr=[v + flat_bump for v in FLAT_IFR],
    )
    return qfCreateModel(value_date, 'YIELD_CURVE', dc, build_method_collection)


VALUE_DATE = '2026-07-17'
yc = build_model(VALUE_DATE)

sofr_index = IndexRegistry().get('SOFR-1B')
sofr_composite = IndexRegistry().get('USD-SOFR-COMPOUND')
libor_3m = IndexRegistry().get('USD-LIBOR-BBA-3M')
funding_identifier = FundingIdentifierRegistry().get('SOFR-1B-FLAT')

vpc = ValuationParametersCollection([
    FundingIndexParameter({
        'FUNDING INDEX': 'SOFR-1B-FLAT',
        'CURRENCIES': '',
        'FUNDING INDICES': '',
        'UNDERLYING FUNDING INDEX': '',
    }),
])

print('component_order_    :', yc.component_order_)

# gradient_lengths_ is populated lazily by get_gradient() (harvested off each component's own
# .grad, defaulting to zero-length if never called) -- force it once here with a no-op
# (reset=False, nothing has a live graph yet) read so N_STATE/SLICES below are correct.
yc.get_gradient(reset=False)
print('gradient_lengths_    :', yc.gradient_lengths_)
N_STATE = sum(yc.gradient_lengths_)
print('N_STATE              :', N_STATE)


component_order_    : ['SOFR-1B', 'USD-LIBOR-BBA-3M', 'SOFR-1B-FLAT']
gradient_lengths_    : [7, 7, 3]
N_STATE              : 17


`get_risk`'s gradient vector is ordered per `component_order_` above -- the cells below slice
it into per-component blocks by that same order rather than assuming a fixed layout, and compare
each block only against a finite difference that bumps *that* component's own target (since
`SOFR-1B-FLAT` and `USD-LIBOR-BBA-3M` are both `REFERENCE`d off `SOFR-1B` and solved/calibrated
independently, a bump to one target does not move the others' own state).

In [3]:
def block_slices(yc_model):
    slices, offset = {}, 0
    for name, length in zip(yc_model.component_order_, yc_model.gradient_lengths_):
        slices[name] = slice(offset, offset + length)
        offset += length
    return slices


SLICES = block_slices(yc)
print(SLICES)


{'SOFR-1B': slice(0, 7, None), 'USD-LIBOR-BBA-3M': slice(7, 14, None), 'SOFR-1B-FLAT': slice(14, 17, None)}


## 1. `ValuationEngineProductFixedAccrued`

A single fixed-coupon cashflow, 3M, 4.5% coupon, `ACT/360`, no index -- its only curve
dependency is the `SOFR-1B-FLAT` discounting leg.

In [4]:
EFFECTIVE = Date('2026-08-19')
COUPON = 0.045
NOTIONAL = 10_000_000.0

fa_biz_conv = ql.Following
fa_hol_conv = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
fa_termination = add_period(EFFECTIVE, Period('3M'), fa_biz_conv, fa_hol_conv)

fa_product = ProductFixedAccrued(
    EFFECTIVE, TermOrDate(fa_termination), PayOrReceive.RECEIVE,
    Currency('USD'), NOTIONAL, COUPON, AccrualBasis.new('ACTUAL/360'),
    business_day_convention=fa_biz_conv, holiday_convention=fa_hol_conv,
)
print('effective  :', fa_product.effective_date)
print('termination:', fa_product.termination_date)
print('payment    :', fa_product.payment_date)
print('accrued    :', fa_product.accrued)


effective  : August 19th, 2026
termination: November 19th, 2026
payment    : November 19th, 2026
accrued    : 0.25555555555555554


### 1a. PV vs. closed form, at a fully-forward value date

In [5]:
fa_engine = ValuationEngineProductFixedAccrued(yc, vpc, fa_product, ValuationRequest.PV)
fa_engine.calculate_value()

tau = fa_product.accrued
df_pay = yc.discount_factor(funding_identifier, fa_product.payment_date, calc_grad=True)
expected_pv = NOTIONAL * tau * COUPON * df_pay

print('engine PV  :', _to_float(fa_engine.value))
print('closed PV  :', float(expected_pv.detach()))
assert abs(_to_float(fa_engine.value) - float(expected_pv.detach())) < 1e-6
assert fa_engine.cash == 0.0
print('PASSED')


engine PV  : 113316.99022703551
closed PV  : 113316.99022703551
PASSED


### 1b. `pv01()` exact closed form (PV is affine in the coupon)

In [6]:
pv01 = fa_engine.pv01()
expected_pv01 = NOTIONAL * tau * float(df_pay.detach()) * 1e-4
print('engine pv01 :', pv01, ' expected:', expected_pv01)
assert abs(pv01 - expected_pv01) < 1e-8
print('PASSED')


engine pv01 : 251.8155338378567  expected: 251.8155338378567
PASSED


### 1c. Settlement-date branches: on payment date (cash realized), and fully matured

In [7]:
yc_on_pay = build_model(fa_product.payment_date)
fa_on_pay = ValuationEngineProductFixedAccrued(yc_on_pay, vpc, fa_product, ValuationRequest.PV)
fa_on_pay.calculate_value()
print('value_date == payment_date -> value:', fa_on_pay.value, ' cash:', fa_on_pay.cash, ' df:', fa_on_pay.df_)
assert fa_on_pay.df_ == 1.0 and fa_on_pay.value == fa_on_pay.cash and fa_on_pay.cash != 0.0

matured_date = add_period(fa_product.payment_date, Period('1D'), fa_biz_conv, fa_hol_conv)
yc_matured = build_model(matured_date)
fa_matured = ValuationEngineProductFixedAccrued(yc_matured, vpc, fa_product, ValuationRequest.PV)
fa_matured.calculate_value()
print('value_date >  payment_date -> value:', fa_matured.value, ' cash:', fa_matured.cash)
assert fa_matured.value == 0.0 and fa_matured.cash == 0.0
print('PASSED')


value_date == payment_date -> value: 115000.0  cash: 115000.0  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 1d. `get_risk()` vs. finite difference

`USD-LIBOR-BBA-3M` never enters a fixed cashflow at all, so that block must be exactly zero.
`SOFR-1B-FLAT` is `REFERENCE`d off `SOFR-1B` (its discount factor is `DF(SOFR-1B) *
DF(own zero-spread state)`), so discounting through `SOFR-1B-FLAT` alone still leaves a real,
nonzero analytic gradient on the `SOFR-1B` block too -- **not** a bug, and not something to
assert away as zero (a wrong assumption caught by this very check on the first pass writing
this notebook). Both nonzero blocks are checked against their own independent finite
difference, bumping one component's target at a time (see the shared-setup note above for why
that matters on a `REFERENCE`d + `BRENT`-solved curve).

In [8]:
EPS = 1e-6
REL_TOL = 1e-4

grad = np.zeros(N_STATE)
fa_engine.get_risk(gradient=grad)
print('SOFR-1B block     sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M   sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(fa_engine.value)


def bumped_pv(engine_cls, product, **bumps):
    yc_bumped = build_model(VALUE_DATE, **bumps)
    eng = engine_cls(yc_bumped, vpc, product, ValuationRequest.PV)
    eng.calculate_value()
    return _to_float(eng.value)


fd_sofr = (bumped_pv(ValuationEngineProductFixedAccrued, fa_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductFixedAccrued, fa_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (bump SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (bump SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block     sum: -38807.18843391627
USD-LIBOR-BBA-3M   sum: 0.0
SOFR-1B-FLAT block sum: -38807.18843391627
finite-diff (bump SOFR-1B)     : -38807.18178697862  vs analytic: -38807.18843391627
finite-diff (bump SOFR-1B-FLAT): -38807.18178697862  vs analytic: -38807.18843391627
PASSED


## 2. Atomic index cashflow 1/3 -- `ValuationEngineProductOvernightIndexCompositeCashflow`

SOFR compounded daily over a 3M period (`USD-SOFR-COMPOUND`, already registered in
`static_files/indices.yaml`), 15bp spread, `leverage=1.5`.

In [9]:
on_biz_conv = sofr_index.payment_business_day_conv
on_hol_conv = sofr_index.settlement_holiday
on_termination = add_period(EFFECTIVE, Period('3M'), on_biz_conv, on_hol_conv)
ON_SPREAD = 0.0015
ON_LEVERAGE = 1.5

on_product = ProductOvernightIndexCompositeCashflow(
    EFFECTIVE, TermOrDate(on_termination), PayOrReceive.RECEIVE, sofr_composite,
    ON_SPREAD, Currency('USD'), NOTIONAL, ON_LEVERAGE,
    payment_business_day_convention=on_biz_conv, payment_holiday_convention=on_hol_conv,
)
print('leverage:', on_product.leverage, ' spread:', on_product.spread)


leverage: 1.5  spread: 0.0015


### 2a. PV vs. closed form (fully forward)

In [10]:
on_engine = ValuationEngineProductOvernightIndexCompositeCashflow(yc, vpc, on_product, ValuationRequest.PV)
on_engine.calculate_value()

on_tau = on_product.accrued
df_eff = yc.discount_factor(sofr_index, on_product.effective_date, calc_grad=True)
df_term = yc.discount_factor(sofr_index, on_product.termination_date, calc_grad=True)
on_expected_forward = (df_eff / df_term - 1.0) / on_tau
on_df_pay = yc.discount_factor(funding_identifier, on_product.payment_date, calc_grad=True)
on_expected_pv = NOTIONAL * on_tau * (ON_LEVERAGE * on_expected_forward + ON_SPREAD) * on_df_pay

print('engine forward:', float(on_engine.forward_rate_.detach()), ' closed:', float(on_expected_forward.detach()))
print('engine PV     :', _to_float(on_engine.value), ' closed:', float(on_expected_pv.detach()))
assert abs(float(on_engine.forward_rate_.detach()) - float(on_expected_forward.detach())) < 1e-10
assert abs(_to_float(on_engine.value) - float(on_expected_pv.detach())) < 1e-6
print('PASSED')


engine forward: 0.042708817063390664  closed: 0.042708817063390664
engine PV     : 165098.3865335841  closed: 165098.3865335841
PASSED


### 2b. `pv01()` exact closed form -- `d(PV)/d(forward_rate)`, so it scales with `leverage`

In [11]:
on_pv01 = on_engine.pv01()
on_expected_pv01 = ON_LEVERAGE * NOTIONAL * on_tau * float(on_df_pay.detach()) * 1e-4
print('engine pv01:', on_pv01, ' expected:', on_expected_pv01)
assert abs(on_pv01 - on_expected_pv01) < 1e-6
print('PASSED')


engine pv01: 377.723300756785  expected: 377.7233007567851
PASSED


### 2c. Realized (on payment date) and matured branches -- needs daily fixings seeded across the whole period

In [12]:
def seed_daily_fixings(index, start, end, rate, biz_conv, hol_conv):
    name = index.index_name()
    if IndexFixingsManager().exists(name):
        qfRemoveIndexFixings(name)
    IndexFixingsManager()._map.setdefault(name, {})
    dates, d = [], start
    while d < end:
        if hol_conv.isBusinessDay(d):
            dates.append(d.ISO())
        d = Date(d + 1)
    qfInsertIndexFixing(name, dates, [rate] * len(dates))


seed_daily_fixings(sofr_index, on_product.effective_date, on_product.termination_date, 0.0430, on_biz_conv, on_hol_conv)

yc_on_pay = build_model(on_product.payment_date)
on_on_pay = ValuationEngineProductOvernightIndexCompositeCashflow(yc_on_pay, vpc, on_product, ValuationRequest.PV)
on_on_pay.calculate_value()
print('value_date == payment_date -> value:', on_on_pay.value, ' cash:', on_on_pay.cash, ' df:', on_on_pay.df_)
assert on_on_pay.df_ == 1.0 and on_on_pay.value == on_on_pay.cash and on_on_pay.cash != 0.0

matured_date = add_period(on_product.payment_date, Period('1D'), on_biz_conv, on_hol_conv)
yc_matured = build_model(matured_date)
on_matured = ValuationEngineProductOvernightIndexCompositeCashflow(yc_matured, vpc, on_product, ValuationRequest.PV)
on_matured.calculate_value()
print('value_date >  payment_date -> value:', on_matured.value, ' cash:', on_matured.cash)
assert on_matured.value == 0.0 and on_matured.cash == 0.0

# fully realized -- pv01/grad_at_par have no live graph left to differentiate
assert on_on_pay.pv01() == 0.0
print('PASSED')


63 fixing(s) for SOFR-1B is(are) inserted.
value_date == payment_date -> value: tensor(169555.7590, dtype=torch.float64)  cash: 169555.75901104786  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 2d. `get_risk()` vs. finite difference -- nonzero on `SOFR-1B` (projection) and `SOFR-1B-FLAT` (discounting), zero on `USD-LIBOR-BBA-3M`

In [13]:
grad = np.zeros(N_STATE)
on_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(on_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductOvernightIndexCompositeCashflow, on_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductOvernightIndexCompositeCashflow, on_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 3709611.3165263617
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -56540.54333341922
finite-diff (SOFR-1B)     : 3709610.5128293857  vs analytic: 3709611.3165263617
finite-diff (SOFR-1B-FLAT): -56540.53366743028  vs analytic: -56540.54333341922
PASSED


## 3. Atomic index cashflow 2/3 -- `ValuationEngineProductIBORIndexCashflow`

`USD-LIBOR-BBA-3M`, a single native-tenor (3M) period, 10bp spread, `leverage=1.0`.

In [14]:
ib_biz_conv = libor_3m.payment_business_day_conv
ib_hol_conv = libor_3m.payment_holiday_conv
ib_termination = add_period(EFFECTIVE, libor_3m.term, ib_biz_conv, ib_hol_conv)
IB_SPREAD = 0.001

ib_product = ProductIBORIndexCashflow(
    EFFECTIVE, TermOrDate(ib_termination), PayOrReceive.RECEIVE, libor_3m,
    IB_SPREAD, Currency('USD'), NOTIONAL,
    payment_business_day_convention=ib_biz_conv, payment_holiday_convention=ib_hol_conv,
)
print('fixing_date:', ib_product.fixing_date, ' accrued:', ib_product.accrued)


fixing_date: August 17th, 2026  accrued: 0.25555555555555554


### 3a. PV vs. closed form (fully forward)

In [15]:
ib_engine = ValuationEngineProductIBORIndexCashflow(yc, vpc, ib_product, ValuationRequest.PV)
ib_engine.calculate_value()

ib_tau_native = accrued(EFFECTIVE, ib_termination, libor_3m.accrual_basis, ib_biz_conv, ib_hol_conv)
df_t0 = yc.discount_factor(libor_3m, EFFECTIVE, calc_grad=True)
df_te = yc.discount_factor(libor_3m, ib_termination, calc_grad=True)
ib_expected_forward = (df_t0 / df_te - 1.0) / ib_tau_native
ib_df_pay = yc.discount_factor(funding_identifier, ib_product.payment_date, calc_grad=True)
ib_tau = ib_product.accrued
ib_expected_pv = NOTIONAL * ib_tau * (ib_expected_forward + IB_SPREAD) * ib_df_pay

print('engine forward:', float(ib_engine.forward_rate_.detach()), ' closed:', float(ib_expected_forward.detach()))
print('engine PV     :', _to_float(ib_engine.value), ' closed:', float(ib_expected_pv.detach()))
assert abs(float(ib_engine.forward_rate_.detach()) - float(ib_expected_forward.detach())) < 1e-10
assert abs(_to_float(ib_engine.value) - float(ib_expected_pv.detach())) < 1e-6
print('PASSED')


engine forward: 0.08790018424108219  closed: 0.08790018424108219
engine PV     : 223864.47352951928  closed: 223864.47352951928
PASSED


### 3b. `pv01()` exact closed form

In [16]:
ib_pv01 = ib_engine.pv01()
ib_expected_pv01 = NOTIONAL * ib_tau * float(ib_df_pay.detach()) * 1e-4
print('engine pv01:', ib_pv01, ' expected:', ib_expected_pv01)
assert abs(ib_pv01 - ib_expected_pv01) < 1e-6
print('PASSED')


engine pv01: 251.8155338378567  expected: 251.8155338378567
PASSED


### 3c. Realized (on payment date) and matured branches -- one fixing at the period's own fixing date

In [17]:
def set_single_fixing(index, date, rate):
    name = index.index_name()
    if IndexFixingsManager().exists(name):
        qfRemoveIndexFixings(name)
    IndexFixingsManager()._map.setdefault(name, {})
    qfInsertIndexFixing(name, [date.ISO()], [rate])


set_single_fixing(libor_3m, ib_product.fixing_date, 0.0455)

yc_on_pay = build_model(ib_product.payment_date)
ib_on_pay = ValuationEngineProductIBORIndexCashflow(yc_on_pay, vpc, ib_product, ValuationRequest.PV)
ib_on_pay.calculate_value()
print('value_date == payment_date -> value:', ib_on_pay.value, ' cash:', ib_on_pay.cash, ' df:', ib_on_pay.df_)
assert ib_on_pay.df_ == 1.0 and ib_on_pay.value == ib_on_pay.cash and ib_on_pay.cash != 0.0

matured_date = add_period(ib_product.payment_date, Period('1D'), ib_biz_conv, ib_hol_conv)
yc_matured = build_model(matured_date)
ib_matured = ValuationEngineProductIBORIndexCashflow(yc_matured, vpc, ib_product, ValuationRequest.PV)
ib_matured.calculate_value()
print('value_date >  payment_date -> value:', ib_matured.value, ' cash:', ib_matured.cash)
assert ib_matured.value == 0.0 and ib_matured.cash == 0.0
assert ib_on_pay.pv01() == 0.0
print('PASSED')


1 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
value_date == payment_date -> value: 118833.33333333333  cash: 118833.33333333333  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 3d. `get_risk()` vs. finite difference

All three blocks are nonzero here: `USD-LIBOR-BBA-3M` (direct projection), `SOFR-1B-FLAT`
(direct discounting), and `SOFR-1B` (indirectly, through *both* `USD-LIBOR-BBA-3M` and
`SOFR-1B-FLAT` being `REFERENCE`d off it -- see the note in 1d above). Each block is checked
against its own independent finite difference.

In [18]:
grad = np.zeros(N_STATE)
ib_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(ib_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductIBORIndexCashflow, ib_product, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductIBORIndexCashflow, ib_product, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductIBORIndexCashflow, ib_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 2462785.544927889
USD-LIBOR-BBA-3M    sum: 2539451.46052019
SOFR-1B-FLAT block  sum: -76665.91559230113
finite-diff (SOFR-1B)         : 2462785.007606726  vs analytic: 2462785.544927889
finite-diff (USD-LIBOR-BBA-3M): 2539451.7819513567  vs analytic: 2539451.46052019
finite-diff (SOFR-1B-FLAT)    : -76665.90248118155  vs analytic: -76665.91559230113
PASSED


## 4. Atomic index cashflow 3/3 -- `ValuationEngineProductIBORCompoundingCashflow`

Three consecutive native 3M `USD-LIBOR-BBA-3M` periods (9M span) geometrically compounded,
5bp spread, `leverage=1.0`. Checked under both `SPREAD_EXCLUSIVE_COMPOUND` and `FLAT_COMPOUND`
-- with no product spread ever reaching the analytics layer (spread is applied once, at the
product level, for both methods), a fully forward-looking period has nothing to distinguish the
two, so they must agree exactly.

In [19]:
cp_termination = add_period(EFFECTIVE, Period('9M'), ib_biz_conv, ib_hol_conv)
CP_SPREAD = 0.0005

cp_products = {}
for method in [CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND, CompoundingMethod.FLAT_COMPOUND]:
    cp_products[method] = ProductIBORCompoundingCashflow(
        EFFECTIVE, TermOrDate(cp_termination), PayOrReceive.RECEIVE, libor_3m,
        CP_SPREAD, Currency('USD'), NOTIONAL, Period('3M'), 1.0,
        pay_date_or_payment_offset=TermOrDate('0D'),
        payment_business_day_convention=ib_biz_conv, payment_holiday_convention=ib_hol_conv,
        compounding_method=method,
    )


### 4a. PV vs. closed form, both compounding methods agree (fully forward)

In [20]:
cur, cp_bounds = EFFECTIVE, []
while cur < cp_termination:
    nxt = add_period(cur, libor_3m.term, ib_biz_conv, ib_hol_conv)
    cp_bounds.append((cur, nxt))
    cur = nxt

cp_df_first = yc.discount_factor(libor_3m, cp_bounds[0][0], calc_grad=True)
cp_df_last = yc.discount_factor(libor_3m, cp_bounds[-1][1], calc_grad=True)
cp_engines = {}
cp_values = {}
for method, product in cp_products.items():
    tau_total = product.accrued
    expected_forward = (cp_df_first / cp_df_last - 1.0) / tau_total
    df_pay = yc.discount_factor(funding_identifier, product.payment_date, calc_grad=True)
    expected_pv = NOTIONAL * tau_total * (expected_forward + CP_SPREAD) * df_pay

    engine = ValuationEngineProductIBORCompoundingCashflow(yc, vpc, product, ValuationRequest.PV)
    engine.calculate_value()
    cp_engines[method] = engine
    cp_values[method] = _to_float(engine.value)

    print(f'--- {method} ---')
    print('engine forward:', float(engine.forward_rate_.detach()), ' closed:', float(expected_forward.detach()))
    print('engine PV     :', _to_float(engine.value), ' closed:', float(expected_pv.detach()))
    assert abs(float(engine.forward_rate_.detach()) - float(expected_forward.detach())) < 1e-8
    assert abs(_to_float(engine.value) - float(expected_pv.detach())) < 1e-4

diff = abs(cp_values[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND] - cp_values[CompoundingMethod.FLAT_COMPOUND])
print('PV difference between methods (should be ~0):', diff)
assert diff < 1e-4
print('PASSED')


--- CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND ---
engine forward: 0.09032092499762352  closed: 0.09032092499762352
engine PV     : 664196.7778209932  closed: 664196.7778209932
--- CompoundingMethod.FLAT_COMPOUND ---
engine forward: 0.09032092499762352  closed: 0.09032092499762352
engine PV     : 664196.7778209932  closed: 664196.7778209932
PV difference between methods (should be ~0): 0.0
PASSED


### 4b. `pv01()` exact closed form, both methods

In [21]:
cp_tau = cp_products[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND].accrued
cp_df_pay = yc.discount_factor(funding_identifier, cp_products[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND].payment_date, calc_grad=True)
cp_expected_pv01 = NOTIONAL * cp_tau * float(cp_df_pay.detach()) * 1e-4

for method, engine in cp_engines.items():
    pv01 = engine.pv01()
    print(f'{method} pv01:', pv01, ' expected:', cp_expected_pv01)
    assert abs(pv01 - cp_expected_pv01) < 1e-4
print('PASSED')


CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND pv01: 731.3257135823853  expected: 731.3257135823853
CompoundingMethod.FLAT_COMPOUND pv01: 731.3257135823853  expected: 731.3257135823853
PASSED


### 4c. Realized (on payment date) and matured branches -- fixings seeded at each sub-period start

In [22]:
cp_sub_starts = [b[0] for b in cp_bounds]
set_single_fixing(libor_3m, cp_sub_starts[0], 0.0450)  # overwritten below to hold all 3
name = libor_3m.index_name()
qfRemoveIndexFixings(name)
IndexFixingsManager()._map.setdefault(name, {})
qfInsertIndexFixing(name, [d.ISO() for d in cp_sub_starts], [0.0450, 0.0452, 0.0455])

cp_product = cp_products[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND]
yc_on_pay = build_model(cp_product.payment_date)
cp_on_pay = ValuationEngineProductIBORCompoundingCashflow(yc_on_pay, vpc, cp_product, ValuationRequest.PV)
cp_on_pay.calculate_value()
print('value_date == payment_date -> value:', cp_on_pay.value, ' cash:', cp_on_pay.cash, ' df:', cp_on_pay.df_)
assert cp_on_pay.df_ == 1.0 and cp_on_pay.value == cp_on_pay.cash and cp_on_pay.cash != 0.0

matured_date = add_period(cp_product.payment_date, Period('1D'), ib_biz_conv, ib_hol_conv)
yc_matured = build_model(matured_date)
cp_matured = ValuationEngineProductIBORCompoundingCashflow(yc_matured, vpc, cp_product, ValuationRequest.PV)
cp_matured.calculate_value()
print('value_date >  payment_date -> value:', cp_matured.value, ' cash:', cp_matured.cash)
assert cp_matured.value == 0.0 and cp_matured.cash == 0.0
assert cp_on_pay.pv01() == 0.0
print('PASSED')


The fixings of USD-LIBOR-BBA-3M are all removed.
1 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
The fixings of USD-LIBOR-BBA-3M are all removed.
3 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
value_date == payment_date -> value: tensor(350725.1389, dtype=torch.float64)  cash: 350725.1389173789  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 4d. `get_risk()` vs. finite difference

Same three-nonzero-block shape as 3d.

In [23]:
cp_engine = cp_engines[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND]
grad = np.zeros(N_STATE)
cp_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(cp_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductIBORCompoundingCashflow, cp_product, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductIBORCompoundingCashflow, cp_product, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductIBORCompoundingCashflow, cp_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 7150289.905834121
USD-LIBOR-BBA-3M    sum: 7707123.368884049
SOFR-1B-FLAT block  sum: -556833.4630499285
finite-diff (SOFR-1B)         : 7150286.559481174  vs analytic: 7150289.905834121
finite-diff (USD-LIBOR-BBA-3M): 7707126.252586022  vs analytic: 7707123.368884049
finite-diff (SOFR-1B-FLAT)    : -556833.2297261804  vs analytic: -556833.4630499285
PASSED


## 5. `ValuationEngineProductInterestRateStream`

A full leg -- a `ValuationEngineProductPortfolio` of per-period cashflows sharing one
`fixed_rate_or_spread`. Two streams: a 1Y quarterly **fixed** leg (4.5% coupon) and a 1Y
quarterly **floating** leg (`USD-SOFR-COMPOUND`, 25bp spread) -- both built off the same
`EFFECTIVE` date used above.

In [24]:
fixed_leg = ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, 0.045,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    accrual_period=Period('3M'),
    accrual_basis=AccrualBasis.new('ACTUAL/360'),
)
float_leg = ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, 0.0025,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    index=sofr_composite,
    accrual_period=Period('3M'),
)
print('fixed leg cashflows  :', fixed_leg.num_cashflows())
print('float leg cashflows  :', float_leg.num_cashflows())


fixed leg cashflows  : 4
float leg cashflows  : 4


### 5a. PV vs. an independent per-cashflow sum, at three different value dates

`value_date` sweeps from before the leg starts (all 4 cashflows forward-looking), to partway
through its life (the first cashflow matured, the rest still forward), to fully after the
leg's last payment (fully matured, PV/cash both zero).

`ValuationEngineProductOvernightIndexCompositeCashflow.calculate_value()` always runs the full
daily-compounding analytics engine first (to get `forward_rate_`) and only *afterward* decides
whether the settlement is zero/cash/discounted based on `value_date_` vs. `payment_date_` --
so even a cashflow that's about to be discarded as "matured" still needs every daily fixing
across its own accrual window seeded, or the fixings lookup itself raises. Seed the whole
floating leg's span with a flat 4.30% daily fixing series upfront to cover every cashflow at
every value date used below.

In [25]:
last_termination = fixed_leg.cashflow(fixed_leg.num_cashflows() - 1).termination_date
seed_daily_fixings(sofr_index, EFFECTIVE, last_termination, 0.0430, on_biz_conv, on_hol_conv)


def manual_leg_pv(model, product, engine_cls):
    total = 0.0
    for prod, weight in product.elements_:
        e = engine_cls(model, vpc, prod, ValuationRequest.PV)
        e.calculate_value()
        total += weight * _to_float(e.value)
    return total


# (i) fully forward
for leg, engine_cls in [(fixed_leg, ValuationEngineProductFixedAccrued),
                         (float_leg, ValuationEngineProductOvernightIndexCompositeCashflow)]:
    eng = ValuationEngineProductInterestRateStream(yc, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    manual = manual_leg_pv(yc, leg, engine_cls)
    print(f'{leg.index.index_name() if leg.index else "FIXED"} leg, fully forward -- engine PV: {_to_float(eng.value):.4f}  manual PV: {manual:.4f}')
    assert abs(_to_float(eng.value) - manual) < 1e-4

# (ii) partway through -- value_date after the first cashflow's payment date but before the
# second's; the first element is matured (contributes 0), the rest are still forward.
first_payment = fixed_leg.cashflow(0).payment_date
mid_date = add_period(first_payment, Period('1D'), on_biz_conv, on_hol_conv)
yc_mid = build_model(mid_date)
for leg, engine_cls in [(fixed_leg, ValuationEngineProductFixedAccrued),
                         (float_leg, ValuationEngineProductOvernightIndexCompositeCashflow)]:
    eng = ValuationEngineProductInterestRateStream(yc_mid, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    manual = manual_leg_pv(yc_mid, leg, engine_cls)
    print(f'{leg.index.index_name() if leg.index else "FIXED"} leg, mid-life -- engine PV: {_to_float(eng.value):.4f}  manual PV: {manual:.4f}')
    assert abs(_to_float(eng.value) - manual) < 1e-4

# (iii) fully matured -- value_date after the last cashflow's payment date
last_payment = fixed_leg.cashflow(fixed_leg.num_cashflows() - 1).payment_date
matured_date = add_period(last_payment, Period('1D'), on_biz_conv, on_hol_conv)
yc_matured_leg = build_model(matured_date)
for leg in [fixed_leg, float_leg]:
    eng = ValuationEngineProductInterestRateStream(yc_matured_leg, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    print(f'{leg.index.index_name() if leg.index else "FIXED"} leg, fully matured -- value: {eng.value}  cash: {eng.cash}')
    assert eng.value == 0.0 and eng.cash == 0.0
print('PASSED')


The fixings of SOFR-1B are all removed.
249 fixing(s) for SOFR-1B is(are) inserted.
FIXED leg, fully forward -- engine PV: 442382.1539  manual PV: 442382.1539
USD-SOFR-COMPOUND leg, fully forward -- engine PV: 447486.4047  manual PV: 447486.4047
FIXED leg, mid-life -- engine PV: 334038.5007  manual PV: 334038.5007
USD-SOFR-COMPOUND leg, mid-life -- engine PV: 336828.1090  manual PV: 336828.1090
FIXED leg, fully matured -- value: 0.0  cash: 0.0
USD-SOFR-COMPOUND leg, fully matured -- value: 0.0  cash: 0.0
PASSED


### 5b. `par_rate_or_spread()` -- solving for the rate/spread that zeros the leg's own PV

For the fixed leg this is trivially `0.0` in isolation (a fixed leg's PV is exactly
proportional to its own coupon, with no other term -- "par" only becomes a non-trivial
quantity relative to a second, offsetting leg, e.g. inside a swap). For the floating leg it's
the genuine par spread level; rebuilding the leg at that spread must zero its PV.

In [26]:
fixed_eng = ValuationEngineProductInterestRateStream(yc, vpc, fixed_leg, ValuationRequest.PV)
fixed_eng.calculate_value()
fixed_par = fixed_eng.par_rate_or_spread()
print('fixed leg par rate (expected exactly 0.0):', fixed_par)
assert fixed_par == 0.0

float_eng = ValuationEngineProductInterestRateStream(yc, vpc, float_leg, ValuationRequest.PV)
float_eng.calculate_value()
float_par = float_eng.par_rate_or_spread()
print('float leg par spread:', float_par)

float_leg_at_par = ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, float_par,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    index=sofr_composite,
    accrual_period=Period('3M'),
)
float_eng_at_par = ValuationEngineProductInterestRateStream(yc, vpc, float_leg_at_par, ValuationRequest.PV)
float_eng_at_par.calculate_value()
print('float leg PV at par spread (should be ~0):', _to_float(float_eng_at_par.value))
assert abs(_to_float(float_eng_at_par.value)) < 1e-3
print('PASSED')


fixed leg par rate (expected exactly 0.0): 0.0
float leg par spread: -0.043019214636875594
float leg PV at par spread (should be ~0): 9.15179043659009e-11
PASSED


### 5c. `pv01()` vs. finite difference (bump the shared fixed_rate/spread by 1bp and reprice the leg)

In [27]:
def leg_pv_at_rate(product_ctor, rate):
    leg = product_ctor(rate)
    eng = ValuationEngineProductInterestRateStream(yc, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    return _to_float(eng.value)


make_fixed_leg = lambda r: ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, r,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    accrual_period=Period('3M'), accrual_basis=AccrualBasis.new('ACTUAL/360'),
)
make_float_leg = lambda r: ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, r,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    index=sofr_composite, accrual_period=Period('3M'),
)

for label, ctor, engine, base_rate in [
    ('fixed', make_fixed_leg, fixed_eng, 0.045),
    ('float', make_float_leg, float_eng, 0.0025),
]:
    pv01 = engine.pv01()
    fd_pv01 = leg_pv_at_rate(ctor, base_rate + 1e-4) - leg_pv_at_rate(ctor, base_rate)
    print(f'{label} leg pv01: {pv01:.4f}   finite-diff: {fd_pv01:.4f}')
    assert abs(pv01 - fd_pv01) < 1e-3
print('PASSED')


fixed leg pv01: 983.0715   finite-diff: 983.0715
float leg pv01: 983.0715   finite-diff: 983.0715
PASSED


### 5d. `get_risk()` (inherited from `ValuationEngineProductPortfolio`) vs. finite difference

`USD-LIBOR-BBA-3M` never enters either leg, so that block must be exactly zero for both. Both
legs discount through `SOFR-1B-FLAT`, which is `REFERENCE`d off `SOFR-1B`, so *both* legs --
including the fixed one -- carry a real nonzero `SOFR-1B` gradient block too (same point as
1d above); the floating leg's `SOFR-1B` block is additionally larger since it also projects
directly off `SOFR-1B`.

In [28]:
for label, leg, engine in [('fixed', fixed_leg, fixed_eng), ('float', float_leg, float_eng)]:
    grad = np.zeros(N_STATE)
    engine.get_risk(gradient=grad)
    print(f'--- {label} leg ---')
    print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
    print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
    print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
    assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

    base_pv = _to_float(engine.value)

    def bumped_leg_pv(**bumps):
        yc_bumped = build_model(VALUE_DATE, **bumps)
        eng = ValuationEngineProductInterestRateStream(yc_bumped, vpc, leg, ValuationRequest.PV)
        eng.calculate_value()
        return _to_float(eng.value)

    fd_sofr = (bumped_leg_pv(sofr_bump=EPS) - base_pv) / EPS
    fd_flat = (bumped_leg_pv(flat_bump=EPS) - base_pv) / EPS
    print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
    print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
    assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
    assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


--- fixed leg ---
SOFR-1B block      sum: -314998.17979925475
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -314998.17979925475
finite-diff (SOFR-1B)     : -314998.05045314133  vs analytic: -314998.17979925475
finite-diff (SOFR-1B-FLAT): -314998.05045314133  vs analytic: -314998.17979925475
--- float leg ---
SOFR-1B block      sum: 9482553.578345941
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -319244.80478431337
finite-diff (SOFR-1B)     : 9482547.957333736  vs analytic: 9482553.578345941
finite-diff (SOFR-1B-FLAT): -319244.67347562313  vs analytic: -319244.80478431337
PASSED


## Summary

In [29]:
print('All tests passed:')
print(' - ValuationEngineProductFixedAccrued: PV, pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductOvernightIndexCompositeCashflow: PV (incl. leverage), pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductIBORIndexCashflow: PV, pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductIBORCompoundingCashflow: PV (both compounding methods), pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductInterestRateStream: PV at 3 value dates, par_rate_or_spread, pv01 vs FD, get_risk vs FD')


All tests passed:
 - ValuationEngineProductFixedAccrued: PV, pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductOvernightIndexCompositeCashflow: PV (incl. leverage), pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductIBORIndexCashflow: PV, pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductIBORCompoundingCashflow: PV (both compounding methods), pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductInterestRateStream: PV at 3 value dates, par_rate_or_spread, pv01 vs FD, get_risk vs FD
